# Cutoff rules: what could the analyst see on `AS_OF`?

Each replay run gets an `AS_OF` date: the last day of data it may see. This notebook loads the
**raw, unfiltered** tables so you can look at them and try a `WHERE` condition per table.
When a condition looks right, copy it into `cutoff_filter()` in `agents/db.py`, then run the last cell.

In [ ]:
import datetime as dt
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AS_OF = dt.date(2025, 12, 14)             # mid flu surge; change it to test other dates
d = f"DATE '{AS_OF.isoformat()}'"         # same {d} you use in agents/db.py

raw = duckdb.connect(":memory:")
TABLES = ["daily_funnel", "daily_supply", "flu_activity", "calendar", "change_log", "markets"]
for t in TABLES:
    raw.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv('{(ROOT / 'data' / (t + '.csv')).as_posix()}')")
print("loaded:", ", ".join(TABLES), "| AS_OF =", AS_OF)

## 1. Look at the tables

In [ ]:
DATE_COL = {"daily_funnel": "date", "daily_supply": "date", "flu_activity": "week_start",
            "calendar": "date", "change_log": "event_date", "markets": None}
for t in TABLES:
    c = DATE_COL[t]
    span = raw.sql(f"SELECT min({c}) AS first, max({c}) AS last, count(*) AS rows FROM {t}").df() if c \
        else raw.sql(f"SELECT count(*) AS rows FROM {t}").df()
    print(f"\n{t}  ({c or 'no date column'})")
    display(span)
    display(raw.sql(f"SELECT * FROM {t} LIMIT 3").df())

## 2. Try a condition

`try_condition(table, condition)` shows how many rows the condition keeps and the date range left.
Use `None` to keep everything.

In [ ]:
def try_condition(table, condition):
    c = DATE_COL[table]
    where = f"WHERE {condition}" if condition else ""
    cols = f"min({c}) AS first_kept, max({c}) AS last_kept, " if c else ""
    out = raw.sql(f"SELECT {cols}count(*) AS rows_kept, (SELECT count(*) FROM {table}) AS rows_total "
                  f"FROM {table} {where}").df()
    out.insert(0, "condition", condition)
    return out

try_condition("daily_funnel", f"date <= {d}")      # example: edit and rerun

In [ ]:
# your turn: one cell per table, or edit this one
try_condition("change_log", None)

## 3. The flu table needs more thought

A CDC week runs Sunday to Saturday. Its number doesn't exist until the week ends, and CDC
publishes it about 6 days after that. Here are the weeks around `AS_OF` for Georgia (Atlanta):

In [ ]:
raw.sql(f"""
    SELECT week_start,
           week_start + INTERVAL 6 DAY  AS week_end,
           week_start + INTERVAL 12 DAY AS about_published,
           round(wili * 100, 2)         AS wili_pct,
           week_start + INTERVAL 6 DAY  <= {d} AS week_over,
           week_start + INTERVAL 12 DAY <= {d} AS published
    FROM flu_activity
    WHERE region = 'ga' AND week_start BETWEEN {d} - INTERVAL 35 DAY AND {d} + INTERVAL 14 DAY
    ORDER BY week_start
""").df()

## 4. Check your rules

After copying your conditions into `agents/db.py`, rerun these two cells.

In [ ]:
import importlib, sys
sys.path.insert(0, str(ROOT))
import agents.db as db_module
importlib.reload(db_module)
print(db_module.Database(AS_OF).describe())

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_db.py"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout[-3000:] or result.stderr[-3000:])